# 08. Analyses de fond

Ce notebook regroupe des **analyses de robustesse et de sensibilité** :
- Sensibilité au déséquilibre du corpus (sous-échantillonnage stratifié)
- Comparaison systématique Panel B4 vs corpus complet
- Limites et interprétation

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
from config import CORPUS_V3, RESULTS_DIR, BLOC_ORDER, BLOC_COLORS

RES_DIR = Path("../data/results")
RES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Déséquilibre du corpus par bloc

Le corpus est déséquilibré : la Gauche radicale contribue à une part majoritaire des textes. Ce biais affecte les distances cosinus, les fighting words et les conclusions sur la « stabilité » des blocs minoritaires.

In [ ]:
df = pd.read_parquet(CORPUS_V3)
df["date"] = pd.to_datetime(df["date"])
df_valid = df[df["bloc"].isin(BLOC_ORDER)].copy()

n_by_bloc = df_valid.groupby("bloc").size().reindex(BLOC_ORDER)
pct = (n_by_bloc / n_by_bloc.sum() * 100).round(1)
tab = pd.DataFrame({"n": n_by_bloc, "pct": pct})
print(tab.to_string())
n_min = n_by_bloc.min()
print(f"\nBloc le plus sous-représenté : n = {int(n_min)}")

## 2. Sensibilité : sous-échantillonnage stratifié

On sous-échantillonne chaque bloc à n_min textes pour tester la robustesse des résultats (distance cosinus, stance moyen) face au déséquilibre.

In [ ]:
n_min = int(n_by_bloc.min())
np.random.seed(42)
df_balanced = (
    df_valid.groupby("bloc", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), n_min), random_state=42))
)

stance_col = "stance_v3" if "stance_v3" in df_balanced.columns else "stance"
if stance_col in df_balanced.columns:
    mean_balanced = df_balanced.groupby("bloc")[stance_col].mean().reindex(BLOC_ORDER)
    mean_full = df_valid.groupby("bloc")[stance_col].mean().reindex(BLOC_ORDER)
    diff = (mean_balanced - mean_full).round(3)
    comp = pd.DataFrame({"complet": mean_full, "sous_echantillonne": mean_balanced, "ecart": diff})
    print("Stance moyen par bloc : corpus complet vs sous-échantillonné (n_min par bloc)")
    print(comp.to_string())
    print("\n→ Si les écarts sont faibles, les conclusions sont robustes au déséquilibre.")

## 3. Panel B4 vs corpus complet

Le Panel B4 restreint aux députés les plus actifs. Comparaison des distributions de stance.

In [ ]:
panel_path = RESULTS_DIR / "panel_b4.csv"
stance_panel_path = RESULTS_DIR / "stance_panel_vs_complet.csv"

if panel_path.exists():
    panel = pd.read_csv(panel_path)
    n_panel = len(panel)
    print(f"Panel B4 : {n_panel} députés")
    print(panel["bloc"].value_counts().to_string())

if stance_panel_path.exists():
    sp = pd.read_csv(stance_panel_path)
    print("\nComparaison stance : panel vs complet")
    print(sp.head(10).to_string())

## 4. Synthèse des limites

- **Déséquilibre** : analyses de sensibilité (sous-échantillonnage) à interpréter avec prudence.
- **Panel B4** : biais de sélection vers les députés les plus actifs — les conclusions sur les trajectoires portent sur un sous-ensemble.
- **Annotation LLM** : pas de validation humaine — les scores sont un proxy cohérent mais non validé.
- **Event studies** : shift temporel descriptif, pas de design causal (groupe contrôle, tendances parallèles).